In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM CMAHO10", con=connection2)

connection2.close()




In [2]:
base2

,arehoscod,arehosdes,arehosdescor,arehosotorcita,arehosotorrece,estregcod,arehosint,arehosintinm
0,01,CONSULTA EXTERNA,CEXT,1,1,1,0,0
1,02,URGENCIAS / EMERGENCIA,EMER,0,1,1,1,1
2,03,HOSPITALIZACION,HOSP,0,1,1,1,0
3,04,AYUDA AL DIAGNOSTICO,AYUD,1,1,1,0,0
4,05,CENTRO QUIRURGICO,CEQX,0,1,1,1,1
5,06,CENTRO OBSTETRICO,COBS,0,1,1,1,0
6,19,AREAS ADMINISTRATIVAS,ADMS,0,0,1,0,0
7,07,UNIDAD DE CUIDADOS INTENSIVOS,UCI,0,1,1,0,0
8,08,UNIDAD DE CUIDADOS INTERMEDIOS,UCIN,0,1,1,0,0
9,09,HOSPITALIZACION AMBULATORIA,HOAM,0,1,1,0,0


In [3]:
base2.columns

Index(['arehoscod', 'arehosdes', 'arehosdescor', 'arehosotorcita',
       'arehosotorrece', 'estregcod', 'arehosint', 'arehosintinm'],
      dtype='object')

In [4]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



#connection3.execute('CREATE TEMPORARY TABLE tmp_cmaho10 ()')
base2.to_sql(name='tmp_cmaho10', con=engine3, if_exists='replace', index=False)
#df.to_sql(name='temp_table', con=engine, if_exists='replace', index=False, method='multi', chunksize=1000, temporary=True)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmaho10 FALSO CON LO OBTENIDO DEL ORACLE

query="""

ALTER TABLE tmp_cmaho10 
ALTER COLUMN arehoscod  TYPE character(2),
ALTER COLUMN arehosdes  TYPE character(30),
ALTER COLUMN arehosdescor  TYPE character(4),
ALTER COLUMN arehosotorcita  TYPE character(1),
ALTER COLUMN arehosotorrece  TYPE character(1),
ALTER COLUMN estregcod  TYPE character(1),
ALTER COLUMN arehosint  TYPE character(1),
ALTER COLUMN arehosintinm  TYPE character(1);

UPDATE cmaho10 
SET 
arehosdes           = t.arehosdes       ,                 
arehosdescor        = t.arehosdescor    ,                 
arehosotorcita      = t.arehosotorcita  ,                 
arehosotorrece      = t.arehosotorrece  ,                
estregcod           = t.estregcod       ,                 
arehosint           = t.arehosint       ,                
arehosintinm        = t.arehosintinm                    

FROM tmp_cmaho10 t 
WHERE cmaho10.arehoscod = t.arehoscod;

INSERT INTO cmaho10 (arehoscod,arehosdes, arehosdescor, arehosotorcita,arehosotorrece, estregcod, arehosint, arehosintinm) 
SELECT arehoscod,arehosdes, arehosdescor, arehosotorcita,arehosotorrece, estregcod, arehosint, arehosintinm
FROM tmp_cmaho10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM cmaho10 
    WHERE cmaho10.arehoscod = tmp_cmaho10.arehoscod
);


"""

c1= text(query)
connection3.execute(c1)



#BORRAMOS LAS TABLAS
base2 = pd.read_sql_query(f"SELECT * FROM cmaho10", con=connection3)
query2="""
DROP TABLE tmp_cmaho10;
"""

c2= text(query2)
connection3.execute(c2)
connection3.close()



In [6]:


#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_cmaho10', con=engine4, if_exists='replace', index=False)

11

In [7]:
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

query_count_before = f"SELECT COUNT(*) FROM dim_areas"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla dim_areas antes de la inserción: {cant_antes}")

Cantidad de filas en la tabla dim_areas antes de la inserción: 11


In [8]:




#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query="""
UPDATE dim_areas 
SET des_are      = t.arehosdes, 
abr_are       = t.arehosdescor


FROM tmp_cmaho10 t 
WHERE dim_areas.cod_are = t.arehoscod;

INSERT INTO dim_areas (cod_are, des_are, abr_are) 
SELECT arehoscod,arehosdes, arehosdescor
FROM tmp_cmaho10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_areas 
    WHERE dim_areas.cod_are = tmp_cmaho10.arehoscod
);

DROP TABLE tmp_cmaho10;
"""
c1= text(query)
connection4.execute(c1)


In [9]:
query_count_after = f"SELECT COUNT(*) FROM dim_areas"
cant_despues = connection4.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla dim_areas después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla dim_areas después de la inserción: 11
La cantidad de filas insertadas fue de: 0


In [10]:
connection4.close()